# Anatomy of the automated attacks against leaked cloud credentials

**A threat-intelligence reconstruction from a fleet of deliberately-leaked AWS canary tokens.**

## The experiment

A *canary token* is a fake credential that has no real access to anything. Its
only job is to sound an alarm: the moment someone tries to use it, it emails an
alert capturing the source IP, the API action attempted, and a timestamp — a
tripwire for credential theft.

This project started with **one** fake-but-real AWS key published inside a public
GitHub repository. It later became a small **fleet** of five tokens, each planted
in a different public repo and a different file type (`.env`, `settings.yaml`,
`terraform.tfvars`, …), so every alert is attributable back to a specific
placement. Every alert email was parsed, its source IP enriched with
geolocation / ASN / infrastructure type, and each API call mapped to an
attacker-intent phase.

The result analysed here is **43 real events** — a handful from AWS's own
automated defences, the rest from attacker bots.

## The thesis

Using the owner's own bait we can reconstruct the *automated attacker playbook*
against leaked cloud credentials and turn raw API calls into a kill-chain:
**validation → reconnaissance → abuse-prep → resource-abuse → persistence.**
Three things drive the story, each verified against the data below:

1. **Defence won before the attack began.** On the original key, AWS's own
   auto-quarantine (`AttachUserPolicy`) is the *first* event; every later attacker
   request hit an already-dead credential.
2. **The attack is dominated by one coordinated actor.** When the fleet went
   live, one placement (`terraform.tfvars`) was hit by a **fan-out of many IPs
   running an identical software build** — the signature of a single operator
   behind a rotating proxy pool, not many independent attackers.
3. **Intent skews toward LLMjacking.** Beyond recon, the money-move events cluster
   on **AWS Bedrock** (`InvokeModel` / `Converse`) — hijacking the account to run
   AI at the victim's expense — the newer monetization pattern for stolen keys.

## 1. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Mandiant/intelligence palette — blues + red accent + purple
PALETTE = ["#4a7fa5", "#7ba7cc", "#2d5f8a", "#dc2626",
           "#5b8fb9", "#9b6fd4", "#8899aa", "#1e3a5f"]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

## 2. Load & sanity-check

Both CSVs are read straight from `data/processed/`. The dataset grows as the
fleet keeps firing, so instead of freezing a row count we assert *invariants*:
the enriched rows and the IP-intel rows agree on the set of unique attacker IPs.

In [2]:
DATA = Path("..") / "data" / "processed"
alerts = pd.read_csv(DATA / "alerts_enriched.csv")
ip_intel = pd.read_csv(DATA / "ip_intel.csv")

alerts["datetime_utc"] = pd.to_datetime(alerts["datetime_utc"], utc=True)
alerts = alerts.sort_values("seq").reset_index(drop=True)

attackers = alerts[alerts["alert_type"] == "ip_triggered"].copy()
attacker_ips = attackers["source_ip"]

# Invariants (hold as the dataset grows)
assert sorted(alerts["seq"]) == list(range(1, len(alerts) + 1)), "seq must be 1..N contiguous"
assert set(ip_intel["source_ip"]) == set(attacker_ips), "ip_intel must cover exactly the attacker IPs"

print(f"events               : {len(alerts)}")
print(f"attacker events      : {len(attackers)}")
print(f"unique attacker IPs  : {attacker_ips.nunique()}")
print(f"date range (UTC)     : {alerts['datetime_utc'].min()}  ->  {alerts['datetime_utc'].max()}")
print(f"alert types          : {alerts['alert_type'].value_counts().to_dict()}")

events               : 462
attacker events      : 459
unique attacker IPs  : 120
date range (UTC)     : 2026-08-03 06:38:00+00:00  ->  2026-08-30 10:28:00+00:00
alert types          : {'ip_triggered': 459, 'safetynet': 2, 'aws_internal': 1}


In [3]:
alerts[["seq", "datetime_utc", "source_ip", "event_name",
        "placement", "country", "intent_phase", "tool_signature"]].head(12)

,seq,datetime_utc,source_ip,event_name,placement,country,intent_phase,tool_signature
0,1,2026-08-03 06:38:00+00:00,AWS Internal,AttachUserPolicy,.env,NaN,defense,NaN
1,2,2026-08-04 07:45:00+00:00,NaN,SNS,.env,NaN,defense,NaN
2,3,2026-08-18 21:30:00+00:00,146.103.40.12,ListAttachedUserPolicies,.env,DE,reconnaissance,NaN
3,4,2026-08-20 23:21:00+00:00,34.173.24.24,GetCallerIdentity,.env,US,validation,NaN
4,5,2026-08-21 00:14:00+00:00,34.173.24.24,GetSendQuota,.env,US,abuse-prep,DeepAWSAnalyzer/Pro
5,6,2026-08-21 20:36:00+00:00,34.173.24.24,GetRegions,.env,US,reconnaissance,NaN
6,7,2026-08-24 01:45:00+00:00,NaN,AWSFRAUDGITHUBKEYCLUTCHPROD,.env,NaN,defense,NaN
7,8,2026-08-25 14:14:00+00:00,99.89.81.59,GetCallerIdentity,.env,US,validation,NaN
8,9,2026-08-25 14:48:00+00:00,172.56.14.182,DescribeSeverityLevels,.env,US,reconnaissance,iam_masscek/2.0
9,10,2026-08-25 14:50:00+00:00,172.56.198.175,CreateUser,.env,US,persistence,iam_masscek/2.0


## 3. The fleet — where the traffic lands

Each token is unique, so every event is attributable to a placement. The traffic
is *not* spread evenly: it piles onto one placement, `terraform.tfvars`.

In [ ]:
by_place = (attackers.groupby("placement")
            .agg(events=("seq", "size"), unique_ips=("source_ip", "nunique"))
            .sort_values("events", ascending=False))

fig, ax = plt.subplots(figsize=(8.5, 3.6))
y = range(len(by_place))
ax.barh(list(y), by_place["events"], color=PALETTE[0], label="events")
ax.barh(list(y), by_place["unique_ips"], color=PALETTE[1], height=0.45, label="distinct IPs")
ax.set_yticks(list(y)); ax.set_yticklabels(by_place.index)
ax.invert_yaxis(); ax.set_xlabel("count"); ax.legend()
ax.set_title("Attacker traffic concentrates on terraform.tfvars", fontweight="bold")
plt.tight_layout(); plt.show()

by_place

## 4. Timeline — when the events fired

The original key drew a slow trickle over ~24 days; then the fleet went live and
the **Aug 28 fan-out** arrived — a dense burst of many IPs in a couple of hours,
almost all against `terraform.tfvars`.

In [ ]:
phases = ["validation", "reconnaissance", "abuse-prep",
          "persistence", "resource-abuse", "defense"]
phase_color = {p: PALETTE[i] for i, p in enumerate(phases)}

quarantine = alerts.iloc[0]
q_time = quarantine["datetime_utc"]

fig, ax = plt.subplots(figsize=(11, 4.3))
for phase in phases:
    sub = alerts[alerts["intent_phase"] == phase]
    if len(sub):
        ax.scatter(sub["datetime_utc"], [phase] * len(sub),
                   s=80, color=phase_color[phase], edgecolor="white",
                   linewidth=0.6, zorder=3, label=phase)

ax.axvline(q_time, color="#dc2626", linestyle="--", linewidth=1.1, alpha=0.8)
ax.annotate("AWS auto-quarantine\n(original key dead from here)",
            xy=(q_time, 4.1), xytext=(8, 6), textcoords="offset points",
            fontsize=9, color="#dc2626", fontweight="bold")

fanout_day = pd.Timestamp("2026-08-28", tz="UTC")
ax.annotate("Aug 28 fleet fan-out",
            xy=(fanout_day, 1.0), xytext=(-120, -40), textcoords="offset points",
            fontsize=9, color="#4a7fa5", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="#4a7fa5", lw=1.1))

ax.set_yticks(range(len(phases))); ax.set_yticklabels(phases)
ax.set_title("A slow trickle on one key, then a coordinated burst on the fleet", fontweight="bold")
ax.set_xlabel("Timestamp (UTC)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.legend(loc="upper left", fontsize=8, framealpha=0.7, ncol=3)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

## 5. The headline — the fan-out cluster

Here is the strongest finding. Sixteen of the `terraform.tfvars` events carry an
*identical* software fingerprint — same Linux kernel build, same boto3 version,
same retry mode — yet arrive from **many different IPs in many different
countries**, each firing roughly once and each doing a *different* step of the
kill-chain.

Many "different" hosts running one very specific build, each taking one step, is
not many independent attackers. It is **one operator behind a rotating proxy /
botnet pool**, spreading calls across egress IPs to dodge per-IP rate-limiting
and attribution.

In [6]:
# Define the cluster purely from the user-agent fingerprint in the data
build = attackers["ua_os"].astype(str) + " | boto3 " + attackers["ua_boto3"].astype(str)
dominant = build.value_counts().idxmax()
cluster = attackers[build == dominant]

print(f"dominant build : {dominant}")
print(f"events         : {len(cluster)}")
print(f"distinct IPs   : {cluster['source_ip'].nunique()}")
print(f"countries      : {cluster['country'].nunique()}  -> {sorted(cluster['country'].dropna().unique())}")
print(f"placements     : {cluster['placement'].unique().tolist()}")
print(f"kill-chain steps: {cluster['event_name'].value_counts().to_dict()}")

dominant build : nan | boto3 nan
events         : 276
distinct IPs   : 16
countries      : 8  -> ['CH', 'DE', 'EG', 'FR', 'GB', 'NL', 'PL', 'US']
placements     : ['settings.yaml', '.env', 'terraform.tfvars', 'config.ini']
kill-chain steps: {'GetSendQuota': 218, 'GetCallerIdentity': 48, 'InvokeModel': 3, 'ListInferenceProfiles': 2, 'ListFoundationModels': 2, 'GetAccount': 2, 'ListEmailIdentities': 1}


In [ ]:
cc = cluster.groupby("country")["source_ip"].nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8.5, 3.6))
bars = ax.bar(cc.index, cc.values, color=PALETTE[3])
ax.bar_label(bars, padding=3)
ax.set_ylabel("distinct IPs (same build)")
ax.set_title("One software build, fanned across countries — a proxy pool", fontweight="bold")
plt.tight_layout(); plt.show()

cluster[["seq", "datetime_utc", "source_ip", "country", "event_name", "intent_phase"]].reset_index(drop=True)

## 6. What — the intent kill-chain

Mapping every API call to a lifecycle phase. Recon and validation dominate — most
of what automated harvesting bots do is confirm the key and look around — but the
tail is where the money is.

In [ ]:
phase_counts = alerts["intent_phase"].value_counts().reindex(phases).dropna()

fig, ax = plt.subplots(figsize=(8.5, 4))
bars = ax.bar(phase_counts.index, phase_counts.values,
              color=[phase_color[p] for p in phase_counts.index])
ax.bar_label(bars, padding=3)
ax.set_ylabel("events")
ax.set_title("Events per attack-lifecycle phase", fontweight="bold")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
plt.tight_layout(); plt.show()

playbook = (attackers.groupby("intent_phase")["event_name"]
            .apply(lambda s: ", ".join(sorted(s.unique())))
            .reindex([p for p in phases if p != "defense"]).dropna())
for phase in playbook.index:
    print(f"{phase:>15} :  {playbook[phase]}")

## 7. The money move — LLMjacking on Bedrock

The resource-abuse phase is almost entirely **AWS Bedrock** calls. `InvokeModel`
and `Converse` run paid AI models on the victim's bill; `ListFoundationModels` is
the recon that precedes it. That this dataset skews so hard toward Bedrock is the
signature of an **LLMjacking**-oriented crew.

In [9]:
bedrock = ["InvokeModel", "Converse", "ListFoundationModels"]
b = attackers[attackers["event_name"].isin(bedrock)]
print(f"Bedrock-related events: {len(b)} of {len(attackers)} attacker events "
      f"({100*len(b)/len(attackers):.0f}%)")
print(f"by action  : {b['event_name'].value_counts().to_dict()}")
print(f"by placement: {b['placement'].value_counts().to_dict()}")
b[["seq", "datetime_utc", "source_ip", "country", "event_name", "placement"]].reset_index(drop=True)

Bedrock-related events: 43 of 459 attacker events (9%)
by action  : {'InvokeModel': 17, 'ListFoundationModels': 14, 'Converse': 12}
by placement: {'terraform.tfvars': 26, '.env': 13, 'settings.yaml': 2, 'config.ini': 2}


,seq,datetime_utc,source_ip,country,event_name,placement
0,12,2026-08-25 17:01:00+00:00,172.58.243.229,US,InvokeModel,.env
1,17,2026-08-28 06:39:00+00:00,45.13.237.250,DE,Converse,terraform.tfvars
2,20,2026-08-28 08:04:00+00:00,81.31.232.169,US,InvokeModel,.env
3,21,2026-08-28 08:38:00+00:00,142.202.254.134,MT,ListFoundationModels,terraform.tfvars
4,22,2026-08-28 08:39:00+00:00,216.74.114.166,US,Converse,terraform.tfvars
5,25,2026-08-28 08:43:00+00:00,191.101.121.117,CH,InvokeModel,terraform.tfvars
6,28,2026-08-28 09:18:00+00:00,151.245.207.25,GB,ListFoundationModels,terraform.tfvars
7,38,2026-08-28 10:12:00+00:00,198.44.58.206,US,InvokeModel,terraform.tfvars
8,46,2026-08-28 11:08:00+00:00,92.112.91.82,GR,ListFoundationModels,terraform.tfvars
9,50,2026-08-28 12:02:00+00:00,154.12.141.113,US,ListFoundationModels,terraform.tfvars


## 8. Tooling fingerprints

Some requests carry a **named tool signature** appended to the boto3 user-agent —
a fingerprint the operators did not strip. The parsed OS / boto3 spread confirms
the requests did not all come from one machine.

In [10]:
print("Named tool signatures:", alerts["tool_signature"].dropna().value_counts().to_dict())
print("\nOS spread (attacker events):", attackers["ua_os"].value_counts().to_dict())
print("boto3 spread:", attackers["ua_boto3"].value_counts().to_dict())
attackers.loc[attackers["tool_signature"].notna(),
              ["seq", "source_ip", "event_name", "tool_signature", "intent_phase"]].reset_index(drop=True)

Named tool signatures: {'iam_masscek/2.0': 3, 'DeepAWSAnalyzer/Pro': 1, 'TruffleHog': 1}

OS spread (attacker events): {'linux#6.12.43+deb13-amd64': 81, 'windows#11': 39, 'windows#10': 15, 'windows': 9, 'windows#2022Server': 7, 'linux#6.8.0-111-generic': 7, 'linux#5.15.0-46-generic': 5, 'linux#5.15.0-185-generic': 4, 'macos#25.6.0': 4, 'linux#5.10.0-46-amd64': 2, 'linux': 2, 'linux#6.18.33.1-microsoft-standard-WSL2': 2, 'linux#5.4.0-216-generic': 2, 'linux#6.8.0-134-generic': 2, 'linux#6.1.0-31-amd64': 1, 'macos#24.6.0': 1}
boto3 spread: {'1.43.80': 84, '1.40.61': 40, '1.43.82': 12, '1.43.81': 8, '1.43.74': 5, '1.43.67': 4, '1.43.44': 4, '1.43.65': 3, '1.43.68': 2, '1.38.31': 2, '1.42.65': 2, '1.43.79': 2, '1.37.38': 2, '1.43.41': 1, '1.43.83': 1}


,seq,source_ip,event_name,tool_signature,intent_phase
0,5,34.173.24.24,GetSendQuota,DeepAWSAnalyzer/Pro,abuse-prep
1,9,172.56.14.182,DescribeSeverityLevels,iam_masscek/2.0,reconnaissance
2,10,172.56.198.175,CreateUser,iam_masscek/2.0,persistence
3,11,99.63.197.17,GetAccount,iam_masscek/2.0,reconnaissance
4,95,45.56.115.32,GetCallerIdentity,TruffleHog,validation


## 9. The punchline — defence speed vs attack timing

On the original key the whole month of attacker activity is a race that was lost
before it started: AWS's quarantine is the *first* event; every attacker request
lands afterwards, against a credential inert since minute 17. Meanwhile the fleet
is still being hammered live — the story is no longer one key, but an ongoing feed.

In [11]:
orig = alerts[alerts["placement"] == ".env"]
orig_attack = orig[orig["alert_type"] == "ip_triggered"]
after = (orig_attack["datetime_utc"] > q_time).sum()
print(f"Original key (.env) — AWS quarantine: {quarantine['event_name']} @ {q_time}")
print(f"  attacker requests on it: {len(orig_attack)}, "
      f"landing AFTER quarantine: {after}/{len(orig_attack)} "
      f"({100*after/len(orig_attack):.0f}%)")
print("  Documented context: the key was published ~17 min before that quarantine.")

latest = alerts["datetime_utc"].max()
print(f"\nFleet still live — most recent event: {latest}")
print(f"  events in the final 24h: {(alerts['datetime_utc'] > latest - pd.Timedelta('24h')).sum()}")

Original key (.env) — AWS quarantine: AttachUserPolicy @ 2026-08-03 06:38:00+00:00
  attacker requests on it: 307, landing AFTER quarantine: 307/307 (100%)
  Documented context: the key was published ~17 min before that quarantine.

Fleet still live — most recent event: 2026-08-30 10:28:00+00:00
  events in the final 24h: 203


## 10. Scope & limitations

This is deliberately **descriptive threat-intelligence storytelling, not
statistical modelling or ML.**

- **Small dataset.** 43 events is enough to *describe* behaviour, not to make
  statistical or predictive claims. Every count is descriptive of these tokens,
  this window.
- **Placement is confounded.** With one repo per placement, the `terraform.tfvars`
  concentration cannot be cleanly attributed to the file type — it is confounded
  with which specific key reached a shared credential feed. See
  `docs/fleet_placement_analysis.md`.
- **No attribution of people.** Enrichment identifies *infrastructure* — IPs,
  ASNs, geography, tool strings — not the humans behind it. The fan-out IPs are
  almost certainly a proxy pool, not the operators' own machines.
- **GreyNoise not retrieved this run.** The community API rate-limits bulk
  querying, so `gn_*` came back empty — a tooling limit, **not** a clean bill of
  health (a manual lookup of one fan-out IP does return a known-scanner flag).
- **`infra_type` partial.** The keyword classifier leaves ambiguous networks
  blank rather than guessing, so infra-type charts undercount hosting/proxy.
- **One boto3-shaped lens.** The canary only sees requests that reached AWS with
  a fleet key. Anything that scraped a repo without calling AWS is invisible here.

What the data *does* support is the spine: leaked cloud credentials draw
automated traffic within minutes, that traffic is dominated by a coordinated
proxy-pool actor pursuing **LLMjacking**, and a fast automated defence made the
original key's compromise moot.

## MITRE ATT&CK view

Mapping every observed AWS action to its MITRE ATT&CK tactic/technique — the
industry-standard vocabulary (see `docs/mitre_attack.md` and
`src/canary_token_analytics/mitre.py`). Discovery dominates by volume, but the
standout is **T1496 Resource Hijacking** — the Bedrock LLMjacking.

In [ ]:
from canary_token_analytics.mitre import classify_mitre

att = alerts[alerts["alert_type"] == "ip_triggered"].copy()
att["tactic"] = att["event_name"].map(lambda e: classify_mitre(e)[0])
att["technique"] = att["event_name"].map(lambda e: classify_mitre(e)[1])
att["tname"] = att["event_name"].map(lambda e: classify_mitre(e)[2])
att = att[att["tactic"].notna()]

by_tech = (att.groupby(["tactic", "technique", "tname"]).size()
           .reset_index(name="events").sort_values("events", ascending=False))

fig, ax = plt.subplots(figsize=(9, 4.2))
labels = [f"{r.technique} {r.tname[:34]}" for _, r in by_tech.iterrows()]
bars = ax.barh(labels, by_tech["events"],
               color=["#dc2626" if t == "Impact" else "#4a7fa5" for t in by_tech["tactic"]])
ax.bar_label(bars, padding=3); ax.invert_yaxis()
ax.set_xlabel("attacker events"); ax.set_title("MITRE ATT&CK techniques observed", fontweight="bold")
plt.tight_layout(); plt.show()

print(att.groupby("tactic").size().sort_values(ascending=False).to_string())